# Hybrid LSTM + XGBoost (Stacking Ensemble)

This notebook trains an LSTM as a feature extractor and feeds its embeddings to XGBoost.
It is designed for time-based forecasting with a fixed horizon.


> **Kernel note:** Use the Jupyter kernel `PM2.5 Hybrid (py310)` (created via `.venv`) so TensorFlow is available.


In [10]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('tensorflow') is None:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        'tensorflow-macos==2.16.2',
        'tensorflow-metal==1.2.0',
    ])
    print('Installed TensorFlow. Restart the kernel, then re-run the notebook.')
else:
    print('TensorFlow already available in this interpreter.')


TensorFlow already available in this interpreter.


## External preprocessing report

See the preprocessing analysis in:
- `data/analysis/preprocessing/preprocess_report.md`
- script: `data/analysis/preprocessing/generate_preprocess_report.py`


In [11]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

try:
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as exc:
    raise ImportError("TensorFlow/Keras is required. Install with: pip install tensorflow") from exc


In [12]:
import random
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [13]:
DATA_PATH = '../data_file/'
OUTPUT_DIR = './output/hybrid'
os.makedirs(OUTPUT_DIR, exist_ok=True)

HORIZONS = [1, 2, 4, 6, 12, 24]  # forecast horizons in hours
SEQ_LEN = 72  # sequence length in hours
EMBEDDING_DIM = 32
EPOCHS = 50
BATCH_SIZE = 64
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
RANDOM_SEED = 42
FILL_GAPS = True


In [14]:
# Load datasets
air = pd.read_csv(f'{DATA_PATH}hanoi_air_quality_history.csv')
weather = pd.read_csv(f'{DATA_PATH}hanoi_weather_history.csv')
holidays = pd.read_csv(f'{DATA_PATH}hanoi_holidays_aligned.csv')
traffic = pd.read_csv(f'{DATA_PATH}hanoi_traffic_proxy.csv')

air['datetime'] = pd.to_datetime(air['datetime'], format='%Y-%m-%d:%H')
weather['datetime'] = pd.to_datetime(weather['datetime'], format='%Y-%m-%d:%H')
holidays['datetime'] = pd.to_datetime(holidays['datetime'])
traffic['datetime'] = pd.to_datetime(traffic['datetime'])

# Normalize holiday columns
if 'holiday_name' not in holidays.columns:
    holidays['holiday_name'] = ''
if 'is_holiday' not in holidays.columns:
    holidays['is_holiday'] = (holidays['holiday_name'].notna() & (holidays['holiday_name'] != '')).astype(int)
holidays = holidays[['datetime', 'is_holiday', 'holiday_name']]

# Merge datasets
df = pd.merge(air, weather, on='datetime', how='inner')
df = pd.merge(df, holidays, on='datetime', how='left')

traffic_cols = ['datetime', 'congestion_index']
if 'congestion_noise' in traffic.columns:
    traffic_cols.append('congestion_noise')
df = pd.merge(df, traffic[traffic_cols], on='datetime', how='left')

# Align time range to the common overlap
min_dt = max(air['datetime'].min(), weather['datetime'].min(), holidays['datetime'].min(), traffic['datetime'].min())
max_dt = min(air['datetime'].max(), weather['datetime'].max(), holidays['datetime'].max(), traffic['datetime'].max())
df = df[(df['datetime'] >= min_dt) & (df['datetime'] <= max_dt)].copy()

# Sort and optionally fill hourly gaps
df = df.sort_values('datetime').reset_index(drop=True)
if FILL_GAPS:
    full_index = pd.date_range(df['datetime'].min(), df['datetime'].max(), freq='H')
    df = df.set_index('datetime').reindex(full_index).rename_axis('datetime').reset_index()

# Fill missing values
df['is_holiday'] = df['is_holiday'].fillna(0).astype(int)
df['holiday_name'] = df['holiday_name'].fillna('')
df['congestion_index'] = df['congestion_index'].fillna(df['congestion_index'].median())
if 'congestion_noise' in df.columns:
    df['congestion_noise'] = df['congestion_noise'].fillna(0)

# Interpolate numeric features (exclude target to avoid leakage)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
target_cols = ['pm25']
fill_cols = [c for c in numeric_cols if c not in target_cols]
if FILL_GAPS and fill_cols:
    df = df.set_index('datetime')
    df[fill_cols] = df[fill_cols].interpolate(method='time').ffill().bfill()
    df = df.reset_index()

df = df.sort_values('datetime').reset_index(drop=True)
df.head()


/var/folders/nl/mcsjphcd59g353jxh4l500zc0000gn/T/ipykernel_33031/1402437031.py:36: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(df['datetime'].min(), df['datetime'].max(), freq='H')


,datetime,aqi,pm25,pm10,o3,so2,no2,co,temp,app_temp,...,pres,vis,clouds,precip,uv,dewpt,is_holiday,holiday_name,congestion_index,congestion_noise
0,2022-12-31 17:00:00,155.0,59.0,73.8,55.7,62.3,9.0,224.5,14.8,14.8,...,1024.0,10.0,87.0,0.0,0.0,10.0,0,,91.0,-1.76
1,2022-12-31 18:00:00,171.0,71.0,88.8,56.0,59.0,6.0,206.0,14.6,14.6,...,1023.0,10.0,87.0,0.0,0.0,10.2,0,,94.0,-3.43
2,2022-12-31 19:00:00,179.0,77.0,96.3,54.0,58.0,6.0,203.7,14.3,14.3,...,1023.0,10.0,83.0,0.0,0.0,10.7,0,,82.0,-2.11
3,2022-12-31 20:00:00,199.0,92.0,115.0,52.0,57.0,6.0,201.3,14.1,14.1,...,1023.0,10.0,79.0,0.0,0.0,11.0,0,,40.0,-0.18
4,2022-12-31 21:00:00,161.0,64.0,80.0,50.0,56.0,6.0,199.0,13.8,13.8,...,1022.0,10.0,75.0,0.0,0.0,11.5,0,,61.0,-1.52


In [15]:
def engineer_features(df_in):
    df = df_in.copy()

    # Time features
    df['hour'] = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month
    df['day'] = df['datetime'].dt.day

    # Cyclic encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Wind direction encoding
    if 'wind_dir' in df.columns:
        df['wind_dir'] = df['wind_dir'] % 360
        df['wind_dir_rad'] = np.deg2rad(df['wind_dir'])
        df['wind_dir_sin'] = np.sin(df['wind_dir_rad'])
        df['wind_dir_cos'] = np.cos(df['wind_dir_rad'])
        df.drop(columns=['wind_dir', 'wind_dir_rad'], inplace=True)

    # Lag features
    lags = [1, 3, 6, 12, 24, 48, 72]
    for lag in lags:
        df[f'pm25_lag_{lag}'] = df['pm25'].shift(lag)
        df[f'temp_lag_{lag}'] = df['temp'].shift(lag)
        df[f'wind_spd_lag_{lag}'] = df['wind_spd'].shift(lag)

    # Rolling stats
    windows = [6, 12, 24, 48]
    for win in windows:
        df[f'pm25_roll_mean_{win}'] = df['pm25'].rolling(win).mean()
        df[f'pm25_roll_std_{win}'] = df['pm25'].rolling(win).std()
        df[f'pm25_roll_min_{win}'] = df['pm25'].rolling(win).min()
        df[f'pm25_roll_max_{win}'] = df['pm25'].rolling(win).max()

    # Differences
    df['pm25_diff_1h'] = df['pm25'].diff(1)
    df['pm25_diff_24h'] = df['pm25'].diff(24)
    df['temp_diff_1h'] = df['temp'].diff(1)

    # Categorical
    df['is_peak_hour'] = df['hour'].apply(lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0)
    df['is_night'] = df['hour'].apply(lambda x: 1 if x >= 22 or x <= 6 else 0)
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

    # Interaction
    df['temp_x_rh'] = df['temp'] * df['rh']
    df['wind_spd_x_temp'] = df['wind_spd'] * df['temp']
    if 'congestion_index' in df.columns:
        df['traffic_x_holiday'] = df['congestion_index'] * df['is_holiday']
        df['traffic_x_peak'] = df['congestion_index'] * df['is_peak_hour']

    # Drop rows with NaN from lag/rolling
    df = df.dropna().reset_index(drop=True)
    return df

df_feat = engineer_features(df)
df_feat.head()


,datetime,aqi,pm25,pm10,o3,so2,no2,co,temp,app_temp,...,pm25_diff_1h,pm25_diff_24h,temp_diff_1h,is_peak_hour,is_night,is_weekend,temp_x_rh,wind_spd_x_temp,traffic_x_holiday,traffic_x_peak
0,2023-01-03 17:00:00,191.0,86.0,107.5,7.3,102.3,52.3,526.8,17.4,17.4,...,-5.0,37.0,-0.2,1,0,0,1426.8,34.800,0.0,72.0
1,2023-01-03 18:00:00,205.0,97.0,121.3,6.0,110.0,52.0,535.7,17.3,17.3,...,11.0,29.0,-0.1,1,0,0,1453.2,34.600,0.0,66.0
2,2023-01-03 19:00:00,224.0,111.0,138.8,8.3,96.0,47.0,467.7,17.0,17.0,...,14.0,27.0,-0.3,1,0,0,1462.0,22.610,0.0,71.0
3,2023-01-03 20:00:00,216.0,105.0,131.3,10.7,82.0,42.0,399.6,16.7,16.7,...,-6.0,37.0,-0.3,0,0,0,1452.9,11.022,0.0,0.0
4,2023-01-03 21:00:00,225.0,112.0,140.0,13.0,68.0,37.0,331.5,16.4,16.4,...,7.0,34.0,-0.3,0,0,0,1459.6,13.120,0.0,0.0


In [16]:
TARGET_COL = 'pm25'
FEATURE_COLS = [c for c in df_feat.columns if c not in ['datetime', 'holiday_name', TARGET_COL]]

def build_sequences(df_in, feature_cols, target_col, seq_len, horizon):
    values = df_in[feature_cols].values
    target = df_in[target_col].values
    dates = df_in['datetime'].values
    X, y, t = [], [], []
    for i in range(seq_len, len(df_in) - horizon + 1):
        X.append(values[i - seq_len:i])
        y.append(target[i + horizon - 1])
        t.append(dates[i + horizon - 1])
    return np.array(X), np.array(y), np.array(t)

def prepare_data(df_in, feature_cols, target_col, seq_len, horizon, train_ratio, val_ratio):
    X, y, t = build_sequences(df_in, feature_cols, target_col, seq_len, horizon)
    n = len(X)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    X_train, y_train, t_train = X[:train_end], y[:train_end], t[:train_end]
    X_val, y_val, t_val = X[train_end:val_end], y[train_end:val_end], t[train_end:val_end]
    X_test, y_test, t_test = X[val_end:], y[val_end:], t[val_end:]

    scaler = StandardScaler()
    X_train_2d = X_train.reshape(-1, X_train.shape[-1])
    X_val_2d = X_val.reshape(-1, X_val.shape[-1])
    X_test_2d = X_test.reshape(-1, X_test.shape[-1])

    scaler.fit(X_train_2d)
    X_train = scaler.transform(X_train_2d).reshape(X_train.shape)
    X_val = scaler.transform(X_val_2d).reshape(X_val.shape)
    X_test = scaler.transform(X_test_2d).reshape(X_test.shape)

    return {
        'X_train': X_train,
        'y_train': y_train,
        't_train': t_train,
        'X_val': X_val,
        'y_val': y_val,
        't_val': t_val,
        'X_test': X_test,
        'y_test': y_test,
        't_test': t_test,
        'scaler': scaler,
    }


In [17]:
def build_lstm_model(seq_len, n_features, embedding_dim=32):
    inputs = keras.Input(shape=(seq_len, n_features))
    x = layers.LSTM(64, return_sequences=True)(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(32, return_sequences=False)(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dense(embedding_dim, activation='relu', name='embedding')(x)
    outputs = layers.Dense(1)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
    return model


In [18]:
results = []
models = {}

for horizon in HORIZONS:
    keras.backend.clear_session()
    data = prepare_data(df_feat, FEATURE_COLS, TARGET_COL, SEQ_LEN, horizon, TRAIN_RATIO, VAL_RATIO)
    X_train = data['X_train']
    y_train = data['y_train']
    t_train = data['t_train']
    X_val = data['X_val']
    y_val = data['y_val']
    t_val = data['t_val']
    X_test = data['X_test']
    y_test = data['y_test']
    t_test = data['t_test']

    model = build_lstm_model(SEQ_LEN, X_train.shape[-1], embedding_dim=EMBEDDING_DIM)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    ]

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )

    # Extract LSTM embeddings and predictions
    embedding_model = keras.Model(model.input, model.get_layer('embedding').output)
    train_embed = embedding_model.predict(X_train, verbose=0)
    val_embed = embedding_model.predict(X_val, verbose=0)
    test_embed = embedding_model.predict(X_test, verbose=0)

    train_pred = model.predict(X_train, verbose=0)
    val_pred = model.predict(X_val, verbose=0)
    test_pred = model.predict(X_test, verbose=0)

    # Use last step features to complement embeddings
    train_last = X_train[:, -1, :]
    val_last = X_val[:, -1, :]
    test_last = X_test[:, -1, :]

    X_train_xgb = np.hstack([train_embed, train_last, train_pred])
    X_val_xgb = np.hstack([val_embed, val_last, val_pred])
    X_test_xgb = np.hstack([test_embed, test_last, test_pred])

    xgb_model = xgb.XGBRegressor(
        n_estimators=2000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=RANDOM_SEED,
        tree_method='hist',
    )

    xgb_model.fit(
        X_train_xgb, y_train,
        eval_set=[(X_val_xgb, y_val)],
        eval_metric='rmse',
        verbose=False,
        early_stopping_rounds=50,
    )

    pred = xgb_model.predict(X_test_xgb)
    rmse = mean_squared_error(y_test, pred, squared=False)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    results.append({
        'horizon': horizon,
        'r2': r2,
        'rmse': rmse,
        'mae': mae,
        'xgb_best_iteration': int(xgb_model.best_iteration) if hasattr(xgb_model, 'best_iteration') else None,
    })

    # Save artifacts
    model.save(os.path.join(OUTPUT_DIR, f'lstm_h{horizon}.keras'))
    xgb_model.save_model(os.path.join(OUTPUT_DIR, f'xgb_h{horizon}.json'))
    pd.DataFrame({
        'datetime': t_test,
        'y_true': y_test,
        'y_pred': pred,
    }).to_csv(os.path.join(OUTPUT_DIR, f'predictions_h{horizon}.csv'), index=False)

    models[horizon] = {
        'lstm': model,
        'xgb': xgb_model,
    }

results_df = pd.DataFrame(results).sort_values('horizon')
results_df.to_csv(os.path.join(OUTPUT_DIR, 'hybrid_results.csv'), index=False)
results_df


Epoch 1/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 13s 39ms/step - loss: 2175.7485 - val_loss: 416.7386 - learning_rate: 0.0010
Epoch 2/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 601.3543 - val_loss: 379.4071 - learning_rate: 0.0010
Epoch 3/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - loss: 417.2367 - val_loss: 388.7693 - learning_rate: 0.0010
Epoch 4/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 37ms/step - loss: 375.4577 - val_loss: 400.5909 - learning_rate: 0.0010
Epoch 5/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 37ms/step - loss: 352.2391 - val_loss: 403.4124 - learning_rate: 0.0010
Epoch 6/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - loss: 320.9961 - val_loss: 419.9307 - learning_rate: 5.0000e-04
Epoch 7/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - loss: 309.0133 - val_loss: 420.1807 - learning_rate: 5.0000e-04
Epoch 8/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - loss: 302.1568 - val_loss: 428.6817 - learning_rate: 5.0000e-04
Epoch 9/50
269/269 ━━━━━━━━━━━━━━━━━━━━ 10s 37ms/step - los

TypeError: XGBModel.fit() got an unexpected keyword argument 'eval_metric'

## Notes
- To run multiple horizons, wrap the sequence build and training in a loop.
- Keep time-based splits to avoid leakage.
- Tune LSTM and XGBoost hyperparameters per horizon.
